In [ ]:
!pip install youtube-transcript-api
!pip install pytube3
!pip install pafy
!pip install youtube_dl
!pip install yt-dlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.4 MB/s eta 0:00:00


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter
import yt_dlp

In [ ]:
def get_video_urls(playlist_url):
    ydl_opts = {
        'quiet': True,
        'extract_flat': True,  # Only extract URLs
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        playlist_info = ydl.extract_info(playlist_url, download=False)
        video_urls = [entry['url'] for entry in playlist_info['entries']]
        video_titles = [entry['title'] for entry in playlist_info['entries']]
        return video_urls, video_titles

In [ ]:

def getTranscript(playlist_url):
  video_urls, video_titles = get_video_urls(playlist_url)

  SPEECH_DICT = {}
  for i in range(0,len(video_urls)):
    url = video_urls[i]
    title = video_titles[i]
    print(title)

    # get video id
    video_id = url.split('https://www.youtube.com/watch?v=')[1]

    SPEECH_DICT[video_id] = {}
    SPEECH_DICT[video_id]['url'] = url
    SPEECH_DICT[video_id]['title'] = title
    try:
      # get all the list of transcripts available in diff languages
      transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)

      # find the one which is in hindi
      transcript = transcript_list.find_transcript(['hi'])

      # translate the transcript to english
      translated_transcript = transcript.translate('en')
      eng_transcript = translated_transcript.fetch()

      # format the transcript to sentence string
      formatter = TextFormatter()
      formatted_transcript = formatter.format_transcript(eng_transcript).replace('\n',' ').replace('[appreciation]','').replace('[music]','')

      # append in the list of speeches
      SPEECH_DICT[video_id]['text'] = formatted_transcript
    except:
      try:
        # get all the list of transcripts available in diff languages
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)

        # find the one which is in hindi
        transcript = transcript_list.find_transcript(['en'])

        # translate the transcript to english
        translated_transcript = transcript.translate('en')
        eng_transcript = translated_transcript.fetch()

        # format the transcript to sentence string
        formatter = TextFormatter()
        formatted_transcript = formatter.format_transcript(eng_transcript).replace('\n',' ').replace('[appreciation]','').replace('[music]','')

      # append in the list of speeches
        SPEECH_DICT[video_id]['text'] = formatted_transcript
      except:
        SPEECH_DICT[video_id]['text'] = 'No Transcript Available'

  return SPEECH_DICT

In [ ]:
playlist_urls = ["https://www.youtube.com/playlist?list=PLqtVCj5iilH4Ia-QrpzLPmU-I2kkKbMql",
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH7VO-yhOJk08TDHVenjvEQ3",#independence day speeches
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH7FPEj8iWVe2ZW-ADtOkwjK",#atal bihari vajpayee
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH5vp001YjQNIn7AgmHqwAnx",#apj abdul kalam
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH6EtIep58nXOf84EFXGw7fi",#ram nath kovid
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH5NtsLtRYjjASsY92n9qdp0",#pratibha patil
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH7qQM0desSK3HmwEcUDhuO0",#radha krishana
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH58HBRYbZYlXdRFU4uejG4m",#neelam sanjeeva reddy
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4QghSrtPnTrdgoGZyhcM22",#president speeches
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH6vh2rG-PcuQe_F2OiEyw8F",#morarji desai
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4r0InJt9XYTrEWRGU0ccG6",#lal bahadur shastri
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4yNuJ0c7paHwzh6HZG5ooV",#indira gandhi
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4SkGxWM7k8X6N1F3jZyh-Q",#jawaharlal nehru
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH60r9bvyWNbq5QFcF1z9Xs8",#election speeches
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4vmSDciJZhSRl2oTEJvGMi",#rajiv gandhi
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH4_cmJ9CzdfWD5tCRMWMbml",#pranab mukherjee
                 "https://www.youtube.com/playlist?list=PLqtVCj5iilH6Zfa_OPKixsq7e__wxJTLj",#kr narayan
                 ]

In [ ]:
import six
import pandas as pd

dfs = []
for playlist_url in playlist_urls:
  speech_dict = getTranscript(playlist_url)
  urls = []
  titles = []
  text = []
  for key,val in six.iteritems(speech_dict):
    urls.append(val['url'])
    titles.append(val['title'])
    text.append(val['text'])
  df = pd.DataFrame({'url':urls,'title':titles,'text':text})
  dfs.append(df)


1977 - Morarji Desai Speech | Janta Party Rally
1977 - Atal Bihari Vajpayee Speech | Janta Party Rally
1977 - Then Janta Party President Chandrashekhar speaking in a Rally
1977 - Atal Bihari Vajpayee victory rally Speech | Janta Party Rally
1977 - Babu Jagjiwan Ram Speech | Janta Party Rally
[Private video]
1995 - Jaswant Singh in an election debate on Doordarshan
1988 By-elections | A Discussion
1989- Former Prime Minister I K Gujral in an election discussion on Doordarshan
1995 - Former PM Atal Bihari Vajpayee in an election debate on Doordarshan
1995 - Ram Vilas Paswan in an election debate on Doordarshan
1989 - Anand Sharma talking about booth capturing | General Election
1989 - P.V Narasimha Rao talking about stable government | General Election
1989 - Jaswant Singh talking about booth capturing | General Election
1989 - Harkishan Singh Surjeet talking about booth capturing | General Election
1989 - Yashwant Sinha talking about booth capturing | General Election
1997 -  Inter-parl

India's 76th  Independence Day Celebrations – PM’s address to the Nation - LIVE from the Red Fort.
2014 - PM Narendra Modi on Planning Commission | Red Fort
[Private video]
1998 - Then PM Atal Bihari Vajpayee's Independence Day Speech | Part 2
1998 - Then PM Atal Bihari Vajpayee's Independence Day Speech | Part 1
1979 - Then PM Chaudhary Charan Singh From the ramparts of the Red Fort
2020 - Ram Nath Kovind's Message to the Nation | Eve of Independence Day
2019 - Ram Nath Kovind's Message to the Nation | Eve of Independence Day
2018 - Ram Nath Kovind's Message to the Nation | Eve of Independence Day
2017 - Ram Nath Kovind's Message to the Nation | Eve of Independence Day
2016 - Pranab Mukherjee's Message to the Nation | Eve of Independence Day
2015 - Pranab Mukherjee's Message to the Nation | Eve of Independence Day
2013 - Pranab Mukherjee's Message to the Nation | Eve of Independence Day
2011 - Pratibha Patil's Message to the Nation | Eve of Independence Day
2010 - Pratibha Patil's Mes

2002 - Atal Bihari Vajpayee's Address to the 57th UN General Assembly
1977 - Then Foreign Minister Atal Bihari Vajpayee as 1st Indian leader to address UNGA in Hindi
1999 में कारगिल युद्ध पर तत्कालीन  प्रधानमंत्री अटल बिहारी वाजपेयी
1998 - Atal Bihari Vajpayee - "Jai Jawan - Jai Kisan - Jai Vigyan" | Red Fort
1977 - Atal Bihari Vajpayee on Emergence of Janata Party
1977 - Atal Bihari Vajpayee address the nation..."राष्ट्रीय जाग्रयम् वयम्"
Atal Bihari Vajpayee speech at Amritsar karyakarta sammelan
Post Kargil victory, Atal Bihari Vajpayee addresses a rally in Chandigarh
After beating Pakistan in Kargil war, Atal Bihari Vajpayee addresses rally in Chennai
2003 Rajasthan Elections - Atal Bihari Vajpayee addresses a rally in Jaipur
2003 Rajasthan Polls - Atal Bihari Vajpayee addresses a rally in Ajmer
2003 Rajasthan Polls - Atal Bihari Vajpayee addresses a rally in Jodhpur
2003 - Then PM Atal Bihari Vajpayee's Independence Day Speech
2002 - Then PM Atal Bihari Vajpayee's Independence Day 

1982 - Then PM Indira Gandhi's visit to USA
Indira Gandhi's visit to Lakshadweep
Indira Gandhi's 20 point program - Special Address to Nation
Indira Gandhi at Deoband - centenary celebrations of Darul Uloom
Visit to Shanti Niketan - circa 1980s
Gandhi Jayanti at Canberra Australia 1981
1981 - UK PM Margaret Thatcher's visit to India
Kasturba Gram Aur Bharat Bhawan   A Report from 1980s
From the Ramparts of Red Fort in 1983: Indira Gandhi’s voice that moved a nation
1982 - Then PM Indira Gandhi's Independence Day Speech
1984 - Then PM Indira Gandhi's Independence day speech
1971 - Then PM Indira Gandhi's Independence Day Speech
1966 - Then PM Indira Gandhi's Independence Day Speech
1970 - Then PM Indira Gandhi's Independence Day Speech
1968 - Then PM Indira Gandhi's Independence Day Speech
1967 - Then PM Indira Gandhi's Independence Day Speech
1975 - Then PM Indira Gandhi's Independence Day Speech
1980 - Then PM Indira Gandhi's Independence Day Speech
1981 - Then PM Indira Gandhi's Inde

Jawaharlal Nehru's last TV Interview - May 1964
1951 - Then PM Jawaharlal Nehru's Independence Day Speech
1952 - Then PM Jawaharlal Nehru's Independence Day Speech
1953 - Then PM Jawaharlal Nehru's Independence Day Speech
Jawaharlal Nehru's interview with Arnold Michaelis - 1958
1954 - Then PM Jawaharlal Nehru's Independence Day Speech
1955 - Then PM Jawaharlal Nehru's Independence Day Speech
1956 - Then PM Jawaharlal Nehru's Independence Day Speech
1957 - Then PM Jawaharlal Nehru's Independence Day Speech
1958 - Then PM Jawaharlal Nehru's Independence Day Speech
1959 - Then PM Jawaharlal Nehru's Independence Day Speech
1960 - Then PM Jawaharlal Nehru's Independence Day Speech
1961 - Then PM Jawaharlal Nehru's Independence Day Speech
1962 - Then PM Jawaharlal Nehru's Independence Day Speech
1947 - Then PM Jawaharlal Nehru's Independence Day Speech at midnight | Tryst with Destiny
Jawaharlal Nehru celebrating his birthday with children
1947 - Jawaharlal Nehru's Constituent Assembly Spee

Documentary on Former PM Morarji Desai's political journey
1998 Rajasthan Assembly Election | Atal Bihari Vajpayee Speech in Bharatpur
1998 General elections - Atal Bihari Vajpayee's Election rally in Bastar
Jawahar Lal Nehru's Broadcast to the Nation ahead of 1951 General Elections
1998 Rajasthan Assembly Election | Atal Bihari Vajpayee Speech in Bharatpur
Documentary on Former PM Morarji Desai's political journey
1952 General Elections - Jawahar Lal Nehru's rally in Guwahati
1984 General Election Broadcast by PV Narsimha Rao
1984 - General Election Broadcast by Atal Bihari Vajpayee
1984 General Election Broadcast by Chaudhary Charan Singh
1977 General Elections - Congress rally at Shajapur in Madhya Pradesh
1998 General Elections - Atal Bihari Vajpayee addresses a rally in Jabalpur
1996 General Elections - Congress rally in Jalandhar
1993 Assembly Elections - Congress rally in Lucknow
1996 General Elections - HD Deve Gowda addresses rally in Varanasi
1999 General Elections - Atal Bih

1989 - Then PM Rajiv Gandhi's independence day speech
1987 - Then PM Rajiv Gandhi's Independence Day Speech
1985 - Then PM Rajiv Gandhi's Independence Day speech | Call for National “Harmony”
1989 Elections - When Rajiv Gandhi promised 'Vikas'
1989 Elections - Rajiv Gandhi on importance of secrecy of military operations
1989 Elections - Rajiv Gandhi slams Pakistan for abetting terrorism in India
1989 Elections - Rajiv Gandhi on Babri Masjid-Ram Mandir dispute
1984 Elections - Rajiv Gandhi questions Opposition's agenda
1984 Polls - Rajiv Gandhi alleges Opposition supporting Tukde Tukde gang
1984 Elections - Rajiv Gandhi in Dehradun talks of "Bharat ke Tukde Tukde" Politics
1984 Polls - Rajiv Gandhi talks about Unity
1984 Polls - Rajiv Gandhi on foreign interference in Indian politics
1984 Elections - Rajiv Gandhi on Scheme benefits not reaching people
1984 Elections - Rajiv Gandhi alleges Opposition trying to disintegrate India
1984 Elections - Rajiv Gandhi questions ideology of then Op

In [ ]:
dataset = pd.concat(dfs)
dataset.drop_duplicates(inplace = True)
dataset = dataset[~dataset.text.str.contains('No Transcript Available')]

In [ ]:
len(dataset)

0

In [ ]:
dataset['text'] = dataset['text'].str.replace('praise','').str.replace('Praise','').str.replace('Music','').str.replace('music','').str.replace('applause','').str.replace('Applause','').str.replace('laughter','').str.replace('Laughter','').str.replace('[','').str.replace(']','')
dataset["words"] = dataset["text"].apply(lambda n: len(n.split()))


In [ ]:
modi_speech = pd.read_csv('PM_Modi_speeches.csv')
modi_speech_en = modi_speech[modi_speech.lang == 'en'][['url','title','text','words']]
dataset = pd.concat([dataset, modi_speech_en])

In [ ]:
len(dataset)

491

In [ ]:
dataset.head()

,url,title,text,words
0,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address in the 15th Episode of ‘Mann Ki B...,"My dear countrymen, Namaskar.\nGenerally, this...",21619
1,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at inauguration of the College an...,Our country’s Agriculture Minister Shri Narend...,10128
2,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at seminar on Atmanirbhar Bharat ...,"My cabinet colleague, Shri Rajnath ji, Chief o...",8497
3,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address to the Nation from the ramparts o...,"My dear countrymen,\nCongratulations and many ...",50260
4,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at the Launch of ‘Transparent Tax...,The process of Structural Reforms going on in ...,11908


In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Required for newer versions of NLTK
from nltk.util import bigrams, trigrams
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import nltk
from nltk.util import bigrams, trigrams
from nltk.tokenize import word_tokenize

# Function to generate bigrams
def generate_bigrams(text):
    tokens = word_tokenize(text)
    return list(bigrams(tokens))

# Function to generate trigrams
def generate_trigrams(text):
    tokens = word_tokenize(text)
    return list(trigrams(tokens))

# Apply functions to the DataFrame
dataset['bigrams'] = dataset['text'].apply(generate_bigrams)
dataset['trigrams'] = dataset['text'].apply(generate_trigrams)

dataset.head()


,url,title,text,words,bigrams,trigrams
0,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address in the 15th Episode of ‘Mann Ki B...,"My dear countrymen, Namaskar.\nGenerally, this...",21619,"[(My, dear), (dear, countrymen), (countrymen, ...","[(My, dear, countrymen), (dear, countrymen, ,)..."
1,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at inauguration of the College an...,Our country’s Agriculture Minister Shri Narend...,10128,"[(Our, country), (country, ’), (’, s), (s, Agr...","[(Our, country, ’), (country, ’, s), (’, s, Ag..."
2,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at seminar on Atmanirbhar Bharat ...,"My cabinet colleague, Shri Rajnath ji, Chief o...",8497,"[(My, cabinet), (cabinet, colleague), (colleag...","[(My, cabinet, colleague), (cabinet, colleague..."
3,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address to the Nation from the ramparts o...,"My dear countrymen,\nCongratulations and many ...",50260,"[(My, dear), (dear, countrymen), (countrymen, ...","[(My, dear, countrymen), (dear, countrymen, ,)..."
4,https://www.pmindia.gov.in/en/news_updates/pms...,PM’s address at the Launch of ‘Transparent Tax...,The process of Structural Reforms going on in ...,11908,"[(The, process), (process, of), (of, Structura...","[(The, process, of), (process, of, Structural)..."


In [ ]:
dataset.to_csv('political_speech_dataset.csv')

In [ ]:
import pandas as pd
dataset = pd.read_csv('political_speech_dataset.csv')

Mounted at /content/drive


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

KeyboardInterrupt: 